# 自动井震标定

本笔记本把 [well_vertical_tdt](notebooks/well_vertical_tdt.ipynb) 的局部时深关系构建流程与 [03_Auto_Well_Tie](notebooks/03_Auto_Well_Tie.ipynb) 的自动标定流程衔接起来。


In [ ]:
import sys
from pathlib import Path
from pprint import pprint

import matplotlib.pyplot as plt
import numpy as np
import yaml

repo_root = Path.cwd().resolve().parent
src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from cup.petrel.export import (
    export_vertical_tdt_to_petrel_checkshots,
    export_vertical_wtie_artifacts,
    format_artifact_tag,
)
from cup.petrel.load import (
    import_interpretation_petrel,
    import_well_heads_petrel,
    import_well_tops_petrel,
    load_vp_rho_logset_from_las,
)
from cup.seismic.process import interpolate_interpretation_surface
from cup.seismic.survey import open_survey
from cup.well.process import clip_logsets_by_well_tops
from wtie.optimize import autotie
from wtie.processing import grid
from wtie.utils import viz
from wtie.utils.datasets import tutorial
from wtie.utils.datasets.utils import InputSet


## 1) 参数区


In [ ]:
well_name = "L9-NW4A"

data_root = repo_root / "data"
well_heads_file = data_root / "raw" / "well_heads"
well_tops_file = data_root / "raw" / "well_tops_new"
las_dir = data_root / "vertical_well_las_target_clear"

top_surface_name = "BVE100_TOP"
base_surface_name = "ITP_BOT"

seismic_file = data_root / "raw" / "mero se 0116_1ms_new_84_coord.Sgy"
top_interpretation_horizon_file = data_root / "interpre_time" / "bve_top_t"
interpretation_offset = 0.0

seismic_iline = 5
seismic_xline = 21
seismic_istep = 1
seismic_xstep = 4

seismic_ctx = open_survey(
    seismic_file,
    seismic_type="segy",
    segy_options={
        "iline": seismic_iline,
        "xline": seismic_xline,
        "istep": seismic_istep,
        "xstep": seismic_xstep,
    },
)

tutorial_folder = data_root / "tutorial"
model_state_dict = tutorial_folder / "trained_net_state_dict.pt"
network_parameters_file = tutorial_folder / "network_parameters.yaml"

output_dir = data_root / "vertical_well_wtie_output"

## 2) 输入检查


In [ ]:
required_paths = [
    well_heads_file,
    well_tops_file,
    las_dir,
    seismic_file,
    top_interpretation_horizon_file,
    model_state_dict,
    network_parameters_file,
]

for p in required_paths:
    assert p.exists(), f"missing required path: {p}"

print("all required inputs exist")

## 3) 导入井数据并裁剪到目标层


In [ ]:
well_heads_df = import_well_heads_petrel(well_heads_file)
well_tops_df = import_well_tops_petrel(well_tops_file)

las_files = sorted(las_dir.glob("*.las"))
logsets = {}
failed = []

for las_path in las_files:
    try:
        logsets[las_path.stem] = load_vp_rho_logset_from_las(las_path)
    except Exception as e:
        failed.append((las_path.name, str(e)))

clip_result = clip_logsets_by_well_tops(
    well_tops_df=well_tops_df,
    logsets=logsets,
    top_surface_name=top_surface_name,
    base_surface_name=base_surface_name,
)
clipped_logsets = clip_result["processed_logsets"]

assert well_name in clipped_logsets, (
    f"well_name={well_name} not found in clipped logsets. available keys: {list(clipped_logsets.keys())[:10]}"
)

print(f"LAS total: {len(las_files)}, loaded: {len(logsets)}, failed: {len(failed)}")
if failed:
    print("failed LAS examples:")
    pprint(failed[:5])

## 4) 井旁道 + 局部时深关系


In [ ]:
row = well_heads_df.loc[well_heads_df["Name"] == well_name]
assert not row.empty, f"well_name={well_name} not found in well_heads"
assert row.shape[0] == 1, f"well_name={well_name} duplicated in well_heads: {row.shape[0]}"

kb = float(row.iloc[0]["Well datum value"])
well_x = float(row.iloc[0]["Surface X"])
well_y = float(row.iloc[0]["Surface Y"])

seismic_trace = seismic_ctx.import_seismic_at_well(
    well_x=well_x,
    well_y=well_y,
    domain="time",
)

interpretation_df = import_interpretation_petrel(top_interpretation_horizon_file)
geometry = seismic_ctx.query_geometry(domain="time")
interp_df = interpolate_interpretation_surface(
    interpretation_df=interpretation_df,
    geometry=geometry,
    outlier_threshold=0.02,
    min_neighbor_count=2,
    keep_nan=True,
)
assert interp_df["interpretation"].notna().all(), "interpolated interpretation contains NaN"

In [ ]:
vp_local = clipped_logsets[well_name].Logs["Vp"]
wp_local = grid.WellPath(vp_local.basis, kb)

il_float, xl_float = seismic_ctx.coord_to_line(x=well_x, y=well_y)

il_min = float(geometry["inline_min"])
xl_min = float(geometry["xline_min"])
il_step = float(geometry["inline_step"])
xl_step = float(geometry["xline_step"])

k_il = (il_float - il_min) / il_step
k_xl = (xl_float - xl_min) / xl_step

k_il0 = int(np.floor(k_il))
k_il1 = int(np.ceil(k_il))
k_xl0 = int(np.floor(k_xl))
k_xl1 = int(np.ceil(k_xl))

il0 = int(round(il_min + k_il0 * il_step))
il1 = int(round(il_min + k_il1 * il_step))
xl0 = int(round(xl_min + k_xl0 * xl_step))
xl1 = int(round(xl_min + k_xl1 * xl_step))

wi = k_il - k_il0
wj = k_xl - k_xl0


def _get_interp_value(il, xl):
    row = interp_df.loc[(interp_df["inline"] == il) & (interp_df["xline"] == xl)]
    assert not row.empty, f"missing interpretation node: inline={il}, xline={xl}"
    v = float(row.iloc[0]["interpretation"])
    assert np.isfinite(v), f"invalid interpretation value: inline={il}, xline={xl}"
    return v


t00 = _get_interp_value(il0, xl0)
t01 = _get_interp_value(il0, xl1)
t10 = _get_interp_value(il1, xl0)
t11 = _get_interp_value(il1, xl1)

interp_value = (1.0 - wi) * (1.0 - wj) * t00 + (1.0 - wi) * wj * t01 + wi * (1.0 - wj) * t10 + wi * wj * t11
origin_time = max(0.0, interp_value + float(interpretation_offset))

local_tdt = grid.TimeDepthTable.get_tvdss_twt_relation_from_vp(
    Vp=vp_local,
    wp=wp_local,
    origin=origin_time,
)

checkshots_file = output_dir / "tdtable_rough" / f"{well_name}_{format_artifact_tag(interpretation_offset)}.txt"
export_vertical_tdt_to_petrel_checkshots(
    output_file=checkshots_file,
    tdt=local_tdt,
    well_name=well_name,
    kb=kb,
    x=well_x,
    y=well_y,
)

fig, ax = plt.subplots(figsize=(4, 5))
ax.plot(local_tdt.twt * 1000.0, local_tdt.tvdss, lw=1.2)
ax.invert_yaxis()
ax.set_xlabel("TWT (ms)")
ax.set_ylabel("TVDSS (m)")
ax.set_title(f"Local Time-Depth: {well_name}")
ax.grid(True, alpha=0.3)
plt.show()

## 5) 组装 InputSet


In [ ]:
inputs = InputSet(
    logset_md=clipped_logsets[well_name],
    seismic=seismic_trace,
    table=local_tdt,
    wellpath=wp_local,
)

print(type(inputs.logset_md), type(inputs.seismic), type(inputs.table), type(inputs.wellpath))

## 6) 加载波子提取器并运行自动标定


In [ ]:
with open(network_parameters_file, "r", encoding="utf-8") as f:
    training_parameters = yaml.load(f, Loader=yaml.Loader)

wavelet_extractor = tutorial.load_wavelet_extractor(training_parameters, model_state_dict)
modeler = tutorial.get_modeling_tool()

search_params = dict(
    num_iters=80,
    similarity_std=0.02,
    suppress_runtime_warnings=True,
    show_all_warnings=False,
)
wavelet_scaling_params = dict(wavelet_min_scale=50000, wavelet_max_scale=500000, num_iters=60)

outputs = autotie.tie_v1(
    inputs,
    wavelet_extractor,
    modeler,
    wavelet_scaling_params,
    search_params=search_params,
    search_space=None,  # type: ignore
    stretch_and_squeeze_params=None,  # type: ignore
)

## 7) QC 与可视化


In [ ]:
best_parameters, values = outputs.ax_client.get_best_parameters()  # type: ignore
means, covariances = values  # type: ignore
print(means)
print(covariances)
pprint(best_parameters)

outputs.plot_optimization_objective();

In [ ]:
fig, axes = outputs.plot_wavelet(fmax=85, phi_max=15, figsize=(6, 5))
axes[0].set_xlim((-0.1, 0.1))
axes[2].set_ylim((-12.5, 12.5))
fig.tight_layout()

In [ ]:
_scale = 120000
fig, axes = outputs.plot_tie_window(wiggle_scale=_scale, figsize=(12.0, 7.5))

In [ ]:
fig, ax = viz.plot_td_table(inputs.table, plot_params=dict(label="initial"))
viz.plot_td_table(outputs.table, plot_params=dict(label="optimized"), fig_axes=(fig, ax))
ax.legend(loc="best")

## 8) 保存井震标定成果与低频模型输入

将优化后的时深表、子波、TWT 测井与低频 AI 曲线统一保存，供后续低频模型构建复用。


In [ ]:
saved_artifacts = export_vertical_wtie_artifacts(
    output_dir=output_dir,
    outputs=outputs,
    well_name=well_name,
    interpretation_offset=interpretation_offset,
    kb=kb,
    x=well_x,
    y=well_y,
)

saved_tdt = outputs.table
saved_ai_log = outputs.logset_twt.AI
saved_wavelet = outputs.wavelet

pprint({key: str(path) for key, path in saved_artifacts.items()})
